# Exam Part 3 - ViDA 10-Day Data Analysis

**Author:** *(your name)*  
**Date:** *(hand-in date)*

We continue to work on the TechDist GmbH — ViDA E-Invoicing Preparation. In exam part 3 we'll look into the compliance status for the 10-day rule.

<br>
<br>

> NOTE: This notebook and exam questions are part of BI-relaterade Programspråk (BI25) at Nackademin. The exam project's setup including it's construct, code and data etc is entirely created for educational purpose and for this course exam. It may not reflect the actual specifics of ViDA, the system data design, integrations, required analysis and required solutions in real world scenario.

## Exam questions, outputs & hand-in

You are provided a Notebook to explore and analyse the ViDA data. 

Opne the OUTLINE below EXPLORE to easily navigate the notebook. 

**EXAM QUESTIONS #** indicates you have a question to answer (code to complete).

At the first line of a code block, when you see <font color=LimeGreen> **# RUN CODE** </font> just read code and comments to understand what it does and then run to see output (if any).

When you see <font color=Magenta> **# YOUR CODD HERE** </font>, please complete the code per requirements.

When you have completed all the exam questions, you are recommended to click on Run All button for the notebook. It should not have any errors. **Save the notebook with outputs. Upload it to StudentPortalen > Examinerad Moment > Inlämning 3: ViDA 10-Day Data Analysis.**



## Grading criteria

You need to complete all `# YOUR CODE HERE` correctly to be able to ``Run All`` to arrive at the final output. 

You will be evaluated for below criteria in order to **Pass (G)**:

1. Code completion
2. Output correctness for exam questions
3. An individual hand-in of this notebook with saved output before deadline on 19 April 2026

To **Pass with distinction (VG)**, you need to complete the exam questions in the VG version of notebook and meet the same criteria as above.

If you fail to hand-in with correct output before deadline on 19 April 2026, you will not be considered passed. An **IG** will be issued. 

## About setting & restoring working directory

In this notebook, we use a simple technique to change your working directory at the beginning and then restore it at the end. Therefore, it's recommended that you **run code blocks one by one** including the last one 0. Restore to original working directory.

If you get stuck mid notebook, it's always ok to reset kernal or reload windows to get you back to your original path.

## 0. Set working directory

We take a simple approach to change your working directory from `notebooks/` to `Exam Project/`and make it easier to fetch data or scripts from parallel folders.

In [ ]:
# RUN CODE

# We store your current working directory so we can restore it at the of this notebook

# your current working directory which should be:
# your computer name path/1 - EXAM/Exam Project/notebooks
# This is the location where THIS particular notebook is

import os

PATH_ORIGINAL = os.getcwd()

In [ ]:
# RUN CODE

from pathlib import Path

# Set working path variable for Exam Project folder which is at the parent level (one level up) from notebooks folder

PATH_PROJECT = Path.cwd().parent

# Change the current working directory

os.chdir(PATH_PROJECT)

In [ ]:
# RUN CODE

# Check current working directory before proceed to next load data step

Path.cwd()

## 1. Load & inspect data

We load in data from **data/raw/** folder.

In [ ]:
# RUN CODE

# Load all the files directly under data/raw/
# we will handle the data/raw/invoice_lines/folder in a moment

import pandas as pd
import numpy as np

path_data = "data/raw/"

files = ["customers", "delivery_addresses", "order_line", "deliveries"]
customers_raw, delivery_addresses_raw, order_line_raw, deliveries_raw = (
    pd.read_csv(Path(path_data) / f"{f}.csv") for f in files
)

In [ ]:
# RUN CODE

# Load all files under data/raw/invoice_lines/
# one new column is added to the raw data file to flag source file name

import glob

dfs = []

for f in Path(path_data + "invoice_lines").glob("*.csv"):
    temp = pd.read_csv(f)
    temp["source_file"] = f.name        # e.g. "invoice_line_2027_02_01.csv"
    dfs.append(temp)

invoice_line_raw = pd.concat(dfs, ignore_index=True)

invoice_line_raw.head(1)

## 📌 EXAM QUESTION 1: function to explore data

You have been provided function name and part of the function code, please complete the rest so the output looks like below. Make sure that you have RUN CODE for the indicated dataframes. Please save the output as part of your individual hand-in.

**EXAMPLE expected output for `explore_data(deliveries_raw)`:**

```
=== DataFrame: deliveries_raw === 

--- shape ---
(433, 4)

--- data types --- 
delivery_confirm_id      str
order_id                 str
delivery_confirm_date    str
delivery_country         str
dtype: object

--- has NaN --- 
delivery_confirm_id      0
order_id                 0
delivery_confirm_date    0
delivery_country         0
dtype: int64

--- describe dataset --- 
       delivery_confirm_id         order_id delivery_confirm_date  \
count                  433              433                   433   
unique                 433              433                   113   
top            CONF-100089  ORD-2026-933206            2027-01-26   
freq                     1                1                    21   

       delivery_country  
count               433  
unique                7  
top             Germany  
freq                105  

--- show top 5 rows --- 
  delivery_confirm_id         order_id delivery_confirm_date delivery_country
0         CONF-100089  ORD-2026-933206            2027-01-20      Netherlands
1         CONF-844144  ORD-2026-400441            2027-01-23           Poland
2         CONF-857501  ORD-2026-676276            2027-01-25           Sweden
3         CONF-601515  ORD-2026-993933            2027-01-24          Belgium
4         CONF-776703  ORD-2026-712546            22-01-2027           Poland

```

In [ ]:
# YOUR CODE HERE

# Complete below function to inspect a dataframe

def explore_data (df: pd.DataFrame):
    '''
    Quick dataframe profiling.
    '''   
    name = [key for key, value in globals().items() if value is df][0]
    print(f"=== DataFrame: {name} === ")
    print("\n--- shape ---")
    print(df.shape)
    # YOUR CODE CONTINUES HERE

In [ ]:
# RUN CODE

# Check if your code shows correctly as required

explore_data(customers_raw)

## 2. Cleanse & transform data

### Copy `_raw` before transformation

In [ ]:
# RUN CODE

# Make copy of all _raw dataframes

# Collect all loaded raw dataframes into a list
dfs = [
    customers_raw,
    delivery_addresses_raw,
    order_line_raw,
    deliveries_raw,
    invoice_line_raw,
]

# Use a generator expression to copy each dataframe, then unpack into named variables
customers, delivery_addresses, order_line, deliveries, invoice_line = (
    df.copy() for df in dfs
)

### Change column name spelling

In [ ]:
# RUN CODE

# Clean up column header spellings into snake_case

# Collect need-to-transform dataframes into a list
dfs = [customers, delivery_addresses]

# for loop with a list comprenhension inside to transform columns names
for df in dfs:
    df.columns = [c.replace(" ", "_").lower() for c in df.columns]

### Create new country_code column

In [ ]:
# RUN CODE

# Inspect unique country values to build the mapping dictionary below

print(customers.groupby("country")["customer_id"].count())
set(customers["country"])

In [ ]:
# RUN CODE

# Create country code mapping

country_code_mapping = {'BE':'BE',
                        'BEL':'BE',
                        'DE':'DE', 
                        'FR':'FR', 
                        'FRA':'FR',
                        'France':'FR',
                         'IT':'IT',
                        'ITA':'IT',
                        'NL':'NL',
                        'NLD':'NL',
                        'Netherlands':'NL',
                        'PL':'PL',
                        'Poland':'PL',
                        'SE':'SE',
                        'SWE':'SE'}

# New column with clean country code

customers["country_code"] = customers["country"].map(country_code_mapping)

In [ ]:
# We use a pandas Boolean Masking to first add a combined condition and name it mask

# Create the condition
mask = ((customers["customer_type"] == "business") & 
        (customers["country_code"] != "DE"))

# Apply it
customers["is_cross_border_b2b"] = mask.map({True: "Yes", False: "No"})

### Clean dates

We clean up a few columns relevant to analyse the 10-day rule.

- order_line["order_date"]
- deliveries["delivery_confirm_date"]
- invoice_line[invoice_date]

You can choose to import a customized function `audit_dates()` and double-check the dates are cleaned and converted to the right datetime type. Please note that this function doesn't clean date, it simply list unique patterns.

In [ ]:
# RUN CODE

# Collect dataframes and their date columns into a list of tuples
date_columns = [
    (order_line, "order_date"),
    (deliveries, "delivery_confirm_date"),
    (invoice_line, "invoice_date"),
]

# Convert date columns to datetime format for each dataframe
for df, col in date_columns:
    df[col] = pd.to_datetime(df[col], format="mixed", dayfirst=True)

## New dataframe `order_delivered_invoiced`

Let's first create a FORWARD-LOOKING new DataFrame `order_delivered_invoiced` based on `order_line` (placed order, to be delivered, to be invoiced) to analyse how the provided datasets meet ViDA's 10-day rules from the perpective of orders.

We keep all the `order_line` columns

- ``order_line_id``
- ``order_id``
- ``order_date``
- ``customer_id``
- ``delivery_place_id``	
- `product_id`
- ``product_description``
- ``product_type``
- ``quantity``
- ``uom``
- ``unit_price``
- ``currency``

And add below columns:

- `country_code`
- `is_cross_border_b2b`
- `delivery_confirm_date`
- `invoice_date`
- `days_order_till_delivery`
- `days_delivery_till_invoice`
- `is_order_delivered`
- `vida_10d_compliant`

We will use below 10-day specifications: 

- 10 days counts from the day after `delivery_confirmed_date`


In [ ]:
# RUN CODE

# Collect dataframe and column into a list of tuples
subset_columns = [
    (customers, ["customer_id", "country_code", "is_cross_border_b2b"]),
    (deliveries, ["order_id", "delivery_confirm_date"]),
    (invoice_line, ["order_id", "invoice_id", "invoice_line_id", "invoice_date"]),]

# Use a generator expression to subset each dataframe to selected columns, then unpack into named variables
customers_sub, deliveries_sub, invoice_line_sub = (
    df[cols] for df, cols in subset_columns
)

# Make sure order_id is unique with invoice_date before joining with order_line
invoice_line_sub = invoice_line_sub[["order_id","invoice_date"]].drop_duplicates()

In [ ]:
# RUN CODE

# Create the new dataframe

order_delivered_invoiced = (
    order_line.merge(customers_sub, on="customer_id", how="left")
    .merge(deliveries_sub, on="order_id", how="left")
    .merge(invoice_line_sub, on="order_id", how="left")
    .assign(
        days_order_till_delivery=lambda df: (
            (df["delivery_confirm_date"] - df["order_date"]).dt.days
        ),
        days_delivery_till_invoice=lambda df: (
            (df["invoice_date"] - df["delivery_confirm_date"]).dt.days
        ),
        is_order_delivered=lambda df: (
            df["days_order_till_delivery"].isna().map({True: "No", False: "Yes"})
        ),
        vida_10d_compliant=lambda df: np.select(
            [
                df["days_order_till_delivery"].isna(),
                df["days_delivery_till_invoice"].isna(),
                df["days_delivery_till_invoice"] < 0,
                (df["days_delivery_till_invoice"] >= 0) & 
                (df["days_delivery_till_invoice"] <= 10),
            ],
            [
                "Not delivered",
                "Not invoiced",  
                "Date problem", 
                "Compliant"
                ],
            default="Not compliant",
        ),
    )
)

In [ ]:
# RUN CODE

# Have a look at how the compliance pattern looks like

(order_delivered_invoiced[["is_cross_border_b2b",
                           "vida_10d_compliant", 
                           "is_order_delivered"]]
                           .drop_duplicates()
                           .sort_values(by=["is_cross_border_b2b", "vida_10d_compliant"], ascending=[False, True])
                           .reset_index(drop=True))

## 📌 EXAM QUESTION 2: BEFORE & AFTER .merge()

Just as what we leanred in SQL course, it's essential to make sure we are not causing duplicates when joining tables - the same is valid for pandas.merge().

So how can we be sure that the new table `order_delivered_invoiced` has been correctly created?

YOUR ANSWER HERE:

Please support your answer with code below:

In [ ]:
# YOUR CODE HERE

## New dataframe ``invoiced_since_delivery``

Then we'll create a BACKWARD-LOOKING new DataFrame `invoiced_since_delivery` based on `invoice_line` (delivered & invoiced) to analyse how the provided datasets meet ViDA's 10-day rules from the perpective of invoices.

We will keep all `invoice_line` columns. Plus some new columns:

- `country_code`
- `is_cross_border_b2b`
- `order_id`
- `delivery_confirm_date`
- `invoice_date`
- `days_delivered_till_invoiced`
- `vida_10d_compliant`

We will use the same specifications: 

- 10 days counts from the day after `delivery_confirmed_date`


In [ ]:
# RUN CODE

# We will need order_line as a connecting table to bring in needed columns from customers table
order_line_sub = order_line[["customer_id", "order_id"]].drop_duplicates()

In [ ]:
# RUN CODE

# Create the new dataframe

invoiced_since_delivery = (
    invoice_line.merge(order_line_sub, on="order_id",how="left")
    .merge(customers_sub, on="customer_id", how="left")
    .merge(deliveries_sub, on="order_id", how="left")
    .assign(
        days_delivered_till_invoiced=lambda df: (
            (df["invoice_date"] - df["delivery_confirm_date"]).dt.days
        ),
        vida_10d_compliant=lambda df: np.select(
            [
                df["days_delivered_till_invoiced"].isna(),
                df["days_delivered_till_invoiced"] < 0,
                (df["days_delivered_till_invoiced"] >= 0) & 
                (df["days_delivered_till_invoiced"] <= 10),
            ],
            [
                "Missing delivery info",  
                "Date problem", 
                "Compliant"
                ],
            default="Not compliant",
        ),
    )
)

## 3. Analyse data

Finally, we will look at a few metrics grouped by selected columns to get an overview on the compliance impact.

In [ ]:
# RUN CODE

# A neat pivot table to dice and slice invoice_line table

invoiced_since_delivery.pivot_table(
    index=["is_cross_border_b2b", "invoice_type", "vida_10d_compliant"],
    values=["line_amount", "vat_amount", "total_amount"],
    aggfunc="sum",
    margins=True,
    margins_name="Total",
    fill_value=0
)[["line_amount", "vat_amount", "total_amount"]]

line_amount  vat_amount  \
is_cross_border_b2b invoice_type vida_10d_compliant                            
Yes                 credit note  Not compliant          -5698.50        0.00   
                    invoice      Not compliant         860890.56        0.00   
                    credit note  Date problem          -17178.80        0.00   
                    invoice      Date problem          459278.93        0.00   
                    credit note  Compliant             -13996.30        0.00   
                    invoice      Compliant             259745.52        0.00   
No                  credit note  Not compliant          -1685.20     -320.19   
                    invoice      Not compliant         719909.59   136782.83   
                                 Date problem          442796.49    84131.33   
                    credit note  Compliant              -8794.20    -1670.90   
                    invoice      Compliant             226991.47    43128.40   

                                                     total_amount  
is_cross_border_b2b invoice_type vida_10d_compliant                
Yes                 credit note  Not compliant           -5698.50  
                    invoice      Not compliant          860890.56  
                    credit note  Date problem           -17178.80  
                    invoice      Date problem           459278.93  
                    credit note  Compliant              -13996.30  
                    invoice      Compliant              259745.52  
No                  credit note  Not compliant           -2005.39  
                    invoice      Not compliant          856692.42  
                                 Date problem           526927.82  
                    credit note  Compliant              -10465.10  
                    invoice      Compliant              270119.87

## 0. Restore to original working directory

Analysis completed. Notebook restores to its original actual path.

In [ ]:
os.chdir(PATH_ORIGINAL)